In [4]:
!pip install numpy pandas matplotlib seaborn scikit-learn tensorflow keras

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

import warnings
warnings.filterwarnings("ignore")

print("All libraries imported successfully!")
print(f"TensorFlow version: ", end="")
import tensorflow as tf
print(tf.__version__)


ModuleNotFoundError: No module named 'tensorflow.python'

In [ ]:
import pandas as pd

# Define the local path (WSL path for your Windows C: drive)
file_path = 'C:/Users/sarga/Things/college/lp5/dl/boston/1_boston_housing.csv' 

# Load the dataset directly from the CSV
data = pd.read_csv(file_path)

# If your CSV already uses 'PRICE', you can skip renaming. 
# If it uses 'MEDV', keep this line:
if 'MEDV' in data.columns:
    data.rename(columns={'MEDV': 'PRICE'}, inplace=True)

print("Dataset loaded successfully from local file!")
print(f"Shape: {data.shape}")
print(f"\nFirst 5 rows:")
print(data.head())

In [ ]:
print("=== Dataset Info ===")
print(data.info())

print("\n=== Statistical Summary ===")
print(data.describe().round(2))

print("\n=== Null Values ===")
print(data.isnull().sum())
print("\nNo null values — dataset is clean!")

In [ ]:
feature_desc = {
    'CRIM':    'Per capita crime rate by town',
    'ZN':      'Proportion of residential land zoned for lots over 25,000 sq.ft.',
    'INDUS':   'Proportion of non-retail business acres per town',
    'CHAS':    'Charles River dummy variable (1 if tract bounds river, 0 otherwise)',
    'NOX':     'Nitric oxides concentration (parts per 10 million)',
    'RM':      'Average number of rooms per dwelling',
    'AGE':     'Proportion of owner-occupied units built prior to 1940',
    'DIS':     'Weighted distances to five Boston employment centres',
    'RAD':     'Index of accessibility to radial highways',
    'TAX':     'Full-value property-tax rate per $10,000',
    'PTRATIO': 'Pupil-teacher ratio by town',
    'B':       '1000(Bk - 0.63)^2 where Bk is the proportion of Black residents',
    'LSTAT':   'Percentage lower status of the population',
    'PRICE':   'Median value of owner-occupied homes in $1000s  ← TARGET'
}

In [ ]:
print("=== Feature Descriptions ===")
for feat, desc in feature_desc.items():
    print(f"  {feat:10s}: {desc}")

In [ ]:
plt.figure(figsize=(14, 4))

plt.subplot(1, 2, 1)
plt.hist(data['PRICE'], bins=30, color='steelblue', edgecolor='black', alpha=0.8)
plt.title('Distribution of House Prices', fontsize=14)
plt.xlabel('Price (in $1000s)')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
sns.boxplot(y=data['PRICE'], color='lightcoral')
plt.title('Box Plot — House Prices', fontsize=14)
plt.ylabel('Price (in $1000s)')

plt.tight_layout()
plt.show()

print(f"Mean Price:   ${data['PRICE'].mean():.2f}k")
print(f"Median Price: ${data['PRICE'].median():.2f}k")
print(f"Std Dev:      ${data['PRICE'].std():.2f}k")
print(f"Min Price:    ${data['PRICE'].min():.2f}k")
print(f"Max Price:    ${data['PRICE'].max():.2f}k")


In [ ]:
plt.figure(figsize=(13, 9))
corr_matrix = data.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # upper triangle mask
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, linewidths=0.5, annot_kws={'size': 8})
plt.title('Correlation Heatmap (Lower Triangle)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
price_corr = corr_matrix['PRICE'].drop('PRICE').sort_values()
print("\n=== Features vs PRICE Correlation ===")
print(price_corr.to_string())
print(f"\nMost NEGATIVE correlation with price: {price_corr.index[0]} ({price_corr.iloc[0]:.3f})")
print(f"Most POSITIVE correlation with price: {price_corr.index[-1]} ({price_corr.iloc[-1]:.3f})")

In [ ]:
plt.figure(figsize=(14, 5))
avg_values = data.drop('PRICE', axis=1).mean(numeric_only=True).sort_values(ascending=False)
colors = sns.color_palette("viridis", len(avg_values))
sns.barplot(x=avg_values.index, y=avg_values.values, palette="viridis")
plt.xticks(rotation=45, ha='right')
plt.title('Average Feature Values', fontsize=14)
plt.ylabel('Mean Value')
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].scatter(data['RM'], data['PRICE'], alpha=0.5, color='steelblue')
axes[0].set_xlabel('RM (Avg Rooms)')
axes[0].set_ylabel('Price ($1000s)')
axes[0].set_title('RM vs Price (r = 0.70)')

axes[1].scatter(data['LSTAT'], data['PRICE'], alpha=0.5, color='tomato')
axes[1].set_xlabel('LSTAT (% Lower Status)')
axes[1].set_ylabel('Price ($1000s)')
axes[1].set_title('LSTAT vs Price (r = -0.74)')

axes[2].scatter(data['NOX'], data['PRICE'], alpha=0.5, color='green')
axes[2].set_xlabel('NOX (Nitric Oxide)')
axes[2].set_ylabel('Price ($1000s)')
axes[2].set_title('NOX vs Price (r = -0.43)')

plt.tight_layout()
plt.show()

In [ ]:
X = data.drop('PRICE', axis=1)
y = data['PRICE']

# 80-20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Training samples:  {X_train.shape[0]}")
print(f"Testing samples:   {X_test.shape[0]}")
print(f"Number of features: {X_train.shape[1]}")
print(f"\nFeature scaling applied: mean ≈ 0, std ≈ 1")
print(f"Scaled X_train mean: {X_train_scaled.mean():.4f}")
print(f"Scaled X_train std:  {X_train_scaled.std():.4f}")

In [ ]:
lin_reg = LinearRegression()
lin_reg.fit(X_train_scaled, y_train)
y_pred_lin = lin_reg.predict(X_test_scaled)

mse_lin = mean_squared_error(y_test, y_pred_lin)
mae_lin = mean_absolute_error(y_test, y_pred_lin)
r2_lin  = r2_score(y_test, y_pred_lin)
rmse_lin = np.sqrt(mse_lin)

print("=== Linear Regression Performance ===")
print(f"  MSE:  {mse_lin:.3f}")
print(f"  RMSE: {rmse_lin:.3f}")
print(f"  MAE:  {mae_lin:.3f}")
print(f"  R²:   {r2_lin:.3f}")


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_lin, alpha=0.7, color='royalblue', label='Predictions')
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Prices ($1000s)', fontsize=12)
plt.ylabel('Predicted Prices ($1000s)', fontsize=12)
plt.title('Linear Regression: Actual vs Predicted Prices', fontsize=14)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
model = Sequential([
    Dense(64, input_dim=X_train_scaled.shape[1], activation='relu'),
    BatchNormalization(),
    Dropout(0.2),

    Dense(32, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),

    Dense(16, activation='relu'),

    Dense(1)  # Output — single value (house price)
], name="BostonDNN")

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

model.summary()


In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=200,
    batch_size=16,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
print(f"\nTraining stopped at epoch: {len(history.history['loss'])}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Train Loss', color='blue')
axes[0].plot(history.history['val_loss'], label='Validation Loss', color='orange')
axes[0].set_title('DNN Model — MSE Loss over Epochs', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['mae'], label='Train MAE', color='blue')
axes[1].plot(history.history['val_mae'], label='Validation MAE', color='orange')
axes[1].set_title('DNN Model — MAE over Epochs', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
y_pred_dnn = model.predict(X_test_scaled).flatten()

mse_dnn  = mean_squared_error(y_test, y_pred_dnn)
mae_dnn  = mean_absolute_error(y_test, y_pred_dnn)
r2_dnn   = r2_score(y_test, y_pred_dnn)
rmse_dnn = np.sqrt(mse_dnn)

print("=== Deep Neural Network Performance ===")
print(f"  MSE:  {mse_dnn:.3f}")
print(f"  RMSE: {rmse_dnn:.3f}")
print(f"  MAE:  {mae_dnn:.3f}")
print(f"  R²:   {r2_dnn:.3f}")


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_dnn, alpha=0.7, color='seagreen', label='Predictions')
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Prices ($1000s)', fontsize=12)
plt.ylabel('Predicted Prices ($1000s)', fontsize=12)
plt.title('DNN: Actual vs Predicted Prices', fontsize=14)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 55)
print(f"{'Model':<25} {'MSE':>7} {'RMSE':>7} {'MAE':>7} {'R²':>7}")
print("-" * 55)
print(f"{'Linear Regression':<25} {mse_lin:>7.3f} {rmse_lin:>7.3f} {mae_lin:>7.3f} {r2_lin:>7.3f}")
print(f"{'Deep Neural Network':<25} {mse_dnn:>7.3f} {rmse_dnn:>7.3f} {mae_dnn:>7.3f} {r2_dnn:>7.3f}")
print("=" * 55)

improvement_mse = ((mse_lin - mse_dnn) / mse_lin) * 100
improvement_r2  = ((r2_dnn - r2_lin) / r2_lin) * 100

print(f"\nDNN reduces MSE by {improvement_mse:.1f}% compared to Linear Regression")
print(f"DNN improves R² by {improvement_r2:.1f}% compared to Linear Regression")
print(f"\nConclusion: DNN significantly outperforms Linear Regression")
print(f"because it captures non-linear relationships between features and price.")
